# LOB Alpha Research — run everything on Google Colab

This notebook clones the repo, downloads the real datasets from Kaggle, and runs every experiment on a Colab GPU,
so nothing heavy runs on your laptop.

**Before you start (one time):**
1. **Runtime → Change runtime type → T4 GPU** (any GPU works).
2. **Kaggle API key:** on kaggle.com go to *Settings → API → Create New Token*. In Colab, open the 🔑 **Secrets**
   panel on the left and add two secrets, `KAGGLE_USERNAME` and `KAGGLE_KEY`, with the values from that
   `kaggle.json`, and allow notebook access.
3. **Optiver data only:** with the same Kaggle account, click *I Understand and Accept* on
   [Realized Volatility rules](https://www.kaggle.com/competitions/optiver-realized-volatility-prediction/rules) and
   [Trading at the Close rules](https://www.kaggle.com/competitions/optiver-trading-at-the-close/rules).
4. **Private repo only:** add a `GITHUB_TOKEN` secret (a GitHub token with read access to the repo).

**Laptop progress carries over.** Section 2b pulls checkpoints the laptop pushed to the `run-state` branch.

**Disconnects are fine.** Results and checkpoints are written to Google Drive
(`MyDrive/ML_Project_2026/artifacts`). Every long run saves after each segment/horizon, so after a disconnect just
run the setup cells again and re-run the experiment cell: it skips finished work.


## 1. Settings

In [ ]:
REPO = "aaravsin-1/ML_Project_2026"   # GitHub owner/name
BRANCH = "master"
SAVE_TO_DRIVE = True                    # keep artifacts + checkpoints on Google Drive
DATA_ON_DRIVE = False                   # raw data on fast local disk (re-downloaded each session, a few minutes)

## 2. Setup: Drive, code, dependencies, Kaggle credentials

In [ ]:
import os, subprocess, sys
from pathlib import Path

if SAVE_TO_DRIVE or DATA_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive/ML_Project_2026")
os.environ["LOB_ARTIFACTS_DIR"] = str(drive_root / "artifacts" if SAVE_TO_DRIVE else Path("/content/artifacts"))
os.environ["LOB_DATA_DIR"] = str(drive_root / "data" if DATA_ON_DRIVE else Path("/content/data"))
for p in (os.environ["LOB_ARTIFACTS_DIR"], os.environ["LOB_DATA_DIR"]):
    Path(p).mkdir(parents=True, exist_ok=True)

from google.colab import userdata
def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
repo_dir = Path("/content/ML_Project_2026")
if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(repo_dir)], check=True)
os.chdir(repo_dir)

for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    value = secret(name)
    if value:
        os.environ[name] = value
print("Kaggle credentials:", "found" if os.environ.get("KAGGLE_KEY") else "MISSING - add the Colab secrets (see top)")
print("Artifacts ->", os.environ["LOB_ARTIFACTS_DIR"])
print("Data      ->", os.environ["LOB_DATA_DIR"])

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("CUDA available:", torch.cuda.is_available())

## 2b. Resume from earlier progress
Runs on the laptop push their checkpoints and results to the repo's `run-state` branch every 10 minutes
(`scripts/sync_checkpoints.py`). This copies them into the artifacts folder, so every experiment below skips
the segments, horizons, stages and folds that already finished. Files already in your Drive folder are never
overwritten.

In [ ]:
!python scripts/sync_checkpoints.py --restore

## 3. Download datasets
Each dataset is skipped if already present. Optiver downloads fail with a clear message until the rules are accepted; the other experiments still run.

In [ ]:
!python scripts/fetch_datasets.py btc fi2010 optiver_close optiver_rvp

## 4. Tests (about a minute)

In [ ]:
!python -m pytest tests -q

## 5. Experiments
Run the cells you want, in any order. Each writes CSVs and plots under the artifacts folder.

| Cell | What | Rough time on a T4 |
|:---|:---|:---|
| 5a | Synthetic data, Experiments A–D, 5 seeds | ~15 min |
| 5b | Real Coinbase BTC, Experiments A–D, 4 segments of ~3 days | ~1.5–2.5 h |
| 5c | FI-2010 Setup 2 vs published DeepLOB | ~1–2 h |
| 5d | Optiver Trading at the Close | ~30–60 min |
| 5e | Optiver Realized Volatility | ~30–60 min |
| 5f | Leakage / robustness report | ~5 min |


In [ ]:
# 5a. Synthetic (checkpointed per seed)
!python -u -W ignore scripts/run_all_experiments.py --source synthetic --seeds 0 1 2 3 4 --epochs 10

In [ ]:
# 5b. Real BTC (checkpointed per segment: re-run after a disconnect to resume)
!python -u -W ignore scripts/run_all_experiments.py --source btc --segments 4 --epochs 8

In [ ]:
# 5c. FI-2010 benchmark (checkpointed per horizon)
!python -u -W ignore -m src.experiments.fi2010_benchmark --epochs 15

In [ ]:
# 5d. Optiver Trading at the Close
!python -u -W ignore -m src.experiments.optiver_close

In [ ]:
# 5e. Optiver Realized Volatility (feature table is cached after the first run)
!python -u -W ignore -m src.experiments.optiver_rvp

In [ ]:
# 5f. Validation report
!python -u -W ignore scripts/verify_robustness_and_leakage.py --seed 42 --events 60000

## 6. Results

In [ ]:
import pandas as pd
art = Path(os.environ["LOB_ARTIFACTS_DIR"])
tables = {
    "Synthetic, Experiment A": art / "experiment_a_results.csv",
    "BTC, Experiment A": art / "btc" / "experiment_a_results.csv",
    "BTC, Experiment D": art / "btc" / "experiment_d_execution_strategies.csv",
    "FI-2010 Setup 2": art / "fi2010" / "fi2010_setup2_results.csv",
    "Optiver Close holdout": art / "optiver_close" / "close_holdout_results.csv",
    "Optiver RVP forward CV": art / "optiver_rvp" / "rvp_summary.csv",
}
for title, path in tables.items():
    print(f"
=== {title} ===")
    print(pd.read_csv(path).round(4).to_string(index=False) if path.exists() else "(not run yet)")

## 7. Take the results home
Downloads a zip of the artifacts folder. Unzip it into the repo's `artifacts/` folder on your laptop
(the per-run `sessions/` checkpoints are left out).

In [ ]:
import shutil
from google.colab import files
staging = Path("/content/artifacts_export")
shutil.rmtree(staging, ignore_errors=True)
shutil.copytree(art, staging, ignore=shutil.ignore_patterns("sessions", "*.pkl"))
zip_path = shutil.make_archive("/content/artifacts", "zip", staging)
files.download(zip_path)

## 8. Optional: push Colab progress back to GitHub
Saves this session's checkpoints to the `run-state` branch, so the laptop (or another Colab session) can resume
from them with `python scripts/sync_checkpoints.py --restore`. Needs a `GITHUB_TOKEN` secret with **write**
access to the repo.

In [ ]:
!git config --global user.email "colab@example.com" && git config --global user.name "Colab runner"
!python scripts/sync_checkpoints.py